# Sesión 2 · Pipelines de datos (ETL) que alimentan la IA
**Databricks AI Engineer** — caso Neptuno

La semana pasada cargamos todo de una vez, a mano. Eso era un **script**.
Hoy construimos un **pipeline**: idempotente, incremental, observable y con calidad declarada.

> 📏 El descuento se aplica **una vez, en silver**. Si vive en cada consulta,
> alguna consulta se lo va a olvidar — y va a ser la que vea el directorio.

## 0 · Conectar con tu catálogo de la sesión 1
Escribe el nombre **completo y exacto** del catálogo que creaste en la S01.
Ejemplo: `neptuno_manuel_arguelles`.

1. Ejecuta la siguiente celda para crear el widget.
2. Escribe el catálogo completo en **Tu catálogo de la S01**.
3. Continúa con la celda de validación.

In [0]:
# Al importar o actualizar el notebook, Databricks no ejecuta el código automáticamente.
# Esta debe ser la primera celda que corras. La llamada directa también vuelve a mostrar
# el widget cuando Databricks conservó su estado interno, pero ocultó la barra tras un Pull.
dbutils.widgets.text("catalogo", "", "Tu catálogo de la S01")

print("✅ Widget creado. Escribe arriba el nombre completo de tu catálogo de la S01.")

✅ Widget creado. Escribe arriba el nombre completo de tu catálogo de la S01.


In [0]:
import re

CATALOGO = dbutils.widgets.get("catalogo").strip().lower()
assert CATALOGO, "Escribe el nombre completo de tu catálogo, por ejemplo: neptuno_manuel_arguelles"
assert CATALOGO.startswith("neptuno_"), (
    "El catálogo debe comenzar por 'neptuno_'. "
    "Escribe el mismo nombre completo que creaste en la S01."
)
assert re.fullmatch(r"[a-z][a-z0-9_]*", CATALOGO), (
    "Usa únicamente letras minúsculas, números y guiones bajos; sin espacios ni tildes."
)

catalogos_disponibles = {fila.catalog.lower() for fila in spark.sql("SHOW CATALOGS").collect()}
assert CATALOGO in catalogos_disponibles, (
    f"No existe el catálogo '{CATALOGO}' en este workspace. "
    "Revisa el nombre en Catalog Explorer y escríbelo exactamente igual."
)

LANDING    = f"/Volumes/{CATALOGO}/bronze/landing"
CHECKPOINT = f"{LANDING}/_checkpoints"
print(f"✅ Catálogo encontrado: {CATALOGO}\nLanding: {LANDING}")

✅ Catálogo encontrado: neptuno_emerson_suarez
Landing: /Volumes/neptuno_emerson_suarez/bronze/landing


## 1 · Silver: la regla de negocio vive acá
`ingreso_linea` se calcula **una sola vez**, con el descuento aplicado.
A partir de este momento, nadie más tiene que acordarse de restarlo.

In [0]:
from pyspark.sql import functions as F

detalles = (
    spark.table(f"{CATALOGO}.bronze.detalles_pedidos")  # Lee la tabla Delta de bronze como DataFrame.
    .withColumn(                                         # Reemplaza PrecioUnidad por una versión tipada.
        "PrecioUnidad",                                 # Columna que se va a crear o reemplazar.
        F.col("PrecioUnidad").cast("decimal(12,2)"),    # Convierte dinero a decimal exacto, nunca float.
    )
    .withColumn(                                         # Reemplaza Cantidad por una versión tipada.
        "Cantidad",                                     # Columna que se va a crear o reemplazar.
        F.col("Cantidad").cast("int"),                  # Convierte la cantidad a número entero.
    )
    .withColumn(                                         # Reemplaza Descuento por una versión tipada.
        "Descuento",                                    # Columna que se va a crear o reemplazar.
        F.col("Descuento").cast("double"),              # Convierte el porcentaje a número decimal.
    )
    .withColumn(                                         # Agrega la métrica de negocio a Silver.
        "ingreso_linea",                                # Nombre único y reutilizable para la métrica.
        F.round(                                         # Redondea el resultado monetario a dos decimales.
            F.col("PrecioUnidad")                       # Precio unitario de la línea de pedido.
            * F.col("Cantidad")                         # Multiplica por las unidades vendidas.
            * (1 - F.col("Descuento")),                 # Aplica el descuento una sola vez, aquí en Silver.
            2,                                           # Conserva dos decimales en el resultado.
        ),
    )
    .drop("_archivo_origen")                             # Quita metadata de S01 que ya no necesita Silver.
)

(
    detalles.write                                       # Abre el escritor batch del DataFrame transformado.
    .mode("overwrite")                                  # Reemplaza la tabla para que la celda sea repetible.
    .saveAsTable(f"{CATALOGO}.silver.detalles_pedidos") # Guarda una tabla Delta registrada en Unity Catalog.
)

spark.sql(f"""
COMMENT ON TABLE {CATALOGO}.silver.detalles_pedidos IS
'Líneas de pedido tipadas. ingreso_linea YA tiene el descuento aplicado: para calcular ventas
 se suma ingreso_linea, nunca PrecioUnidad*Cantidad.'
""")
display(spark.table(f"{CATALOGO}.silver.detalles_pedidos").limit(5))

IdPedido,IdProducto,PrecioUnidad,Cantidad,Descuento,_ingesta_ts,ingreso_linea
10248,11,14.00,12,0.0,2026-08-31T04:50:16.295Z,168.0
10248,42,9.80,10,0.0,2026-08-31T04:50:16.295Z,98.0
10248,72,34.80,5,0.0,2026-08-31T04:50:16.295Z,174.0
10249,14,18.60,9,0.0,2026-08-31T04:50:16.295Z,167.4
10249,51,42.40,40,0.0,2026-08-31T04:50:16.295Z,1696.0


In [0]:
spark.sql(f"""
SELECT COUNT(*) AS filas
FROM {CATALOGO}.silver.detalles_pedidos
""").display()

filas
2155


In [0]:
spark.sql(f"""
SELECT COUNT(*) AS inconsistencias
FROM {CATALOGO}.silver.detalles_pedidos
WHERE ingreso_linea <> ROUND(PrecioUnidad * Cantidad * (1 - Descuento), 2)
""").display()

inconsistencias
0


## 2 · Auto Loader: enterarse solo de lo que llegó nuevo
El **checkpoint** es donde el pipeline recuerda qué archivos ya procesó.
Es lo que hace que correrlo dos veces no duplique nada.

In [0]:
def ingerir_pedidos() -> int:
    """Procesa los archivos nuevos de pedidos/. Devuelve cuántas filas entraron."""
    destino = f"{CATALOGO}.bronze.pedidos_incremental"

    (
        spark.readStream                                  # Crea una lectura incremental con Structured Streaming.
        .format("cloudFiles")                            # Activa Auto Loader para descubrir archivos nuevos.
        .option("cloudFiles.format", "csv")             # Indica que cada archivo descubierto es un CSV.
        .option(                                          # Define dónde Auto Loader guarda el esquema inferido.
            "cloudFiles.schemaLocation",                 # Clave de configuración del almacenamiento de esquema.
            f"{CHECKPOINT}/pedidos_schema",              # Ruta separada del checkpoint de progreso.
        )
        .option(                                          # Decide qué hacer si mañana aparece una columna nueva.
            "cloudFiles.schemaEvolutionMode",            # Clave que controla la evolución del esquema.
            "addNewColumns",                             # Agrega columnas nuevas y las conserva en el destino.
        )
        .option("header", True)                          # Usa la primera fila del CSV como nombres de columnas.
        .load(f"{LANDING}/pedidos/")                     # Observa esta carpeta; no vuelve a leer lo ya procesado.
        .withColumn(                                      # Agrega trazabilidad sobre el archivo de procedencia.
            "_archivo_origen",                           # Nombre de la columna técnica que guardaremos.
            F.col("_metadata.file_path"),                 # Ruta real del CSV entregada por Auto Loader.
        )
        .withColumn(                                      # Agrega trazabilidad temporal de la ingesta.
            "_ingesta_ts",                               # Nombre de la columna técnica de auditoría.
            F.current_timestamp(),                        # Momento en que esta corrida procesó la fila.
        )
        .writeStream                                      # Cambia del lector incremental al escritor incremental.
        .option(                                          # Configura la memoria de progreso del stream.
            "checkpointLocation",                        # Clave obligatoria para una ingesta idempotente.
            f"{CHECKPOINT}/pedidos",                     # Recuerda exactamente qué archivos ya procesó.
        )
        .trigger(availableNow=True)                       # Procesa lo disponible ahora y luego se detiene.
        .toTable(destino)                                 # Escribe incrementalmente en la tabla Delta destino.
        .awaitTermination()                               # Espera a que esta corrida termine antes de continuar.
    )

    return spark.table(destino).count()                   # Cuenta el total acumulado para verificar el efecto.

> 🪤 **Antes de subir nada:** un subdirectorio dentro de un Volume **no se crea solo**.
> Si copiás a `.../landing/pedidos/` sin haberlo creado, falla con `no such directory`.
> Desde el CLI: `databricks fs mkdir dbfs:/Volumes/<cat>/bronze/landing/pedidos`.
> Desde el notebook: `dbutils.fs.mkdirs(f"{LANDING}/pedidos")`.

In [0]:
dbutils.fs.mkdirs(f"{LANDING}/pedidos")
dbutils.fs.mkdirs(f"{LANDING}/detalles")
print("subdirectorios listos")

subdirectorios listos


### Alternativa verificada: `COPY INTO`
Si Auto Loader no arranca (permisos de checkpoint, cluster sin streaming), `COPY INTO`
enseña **exactamente la misma idea** y está medido: 3 meses → 70 pedidos · segunda corrida
→ **0 filas** · cae un mes → entran **26 exactas**.

```sql
COPY INTO <catalogo>.bronze.pedidos_inc
FROM '/Volumes/<catalogo>/bronze/landing/pedidos'
FILEFORMAT = CSV FORMAT_OPTIONS('header'='true')
COPY_OPTIONS('mergeSchema'='true')
```
El registro de archivos ya procesados cumple el mismo papel que el checkpoint.

### Paso 1 — primera corrida (los meses que ya están en el landing)

In [0]:
total_1 = ingerir_pedidos()
print(f"Después de la 1ª corrida: {total_1:,} pedidos")

Después de la 1ª corrida: 22 pedidos


### Paso 2 — se corre otra vez, **sin tocar nada**
Si el total no cambia, el pipeline es idempotente. Ése es el punto entero.

In [0]:
total_2 = ingerir_pedidos()
print(f"Después de la 2ª corrida: {total_2:,} pedidos")
assert total_2 == total_1, "❌ Se duplicaron filas: el checkpoint no está funcionando"
print("✅ Corrió de nuevo y no entró nada. El pipeline es idempotente.")

Después de la 2ª corrida: 22 pedidos
✅ Corrió de nuevo y no entró nada. El pipeline es idempotente.


### Paso 3 — cae un mes más
👉 Sube **un archivo mensual adicional** a `landing/pedidos/` y corre la celda de abajo.
El total debe aumentar solo por las filas de ese archivo; el mes anterior no se reprocesa.

In [0]:
total_3 = ingerir_pedidos()
print(f"Después de que cayeron meses nuevos: {total_3:,} pedidos (+{total_3 - total_2:,})")

Después de que cayeron meses nuevos: 47 pedidos (+25)


### La prueba de que nada se re-escribió
Las filas viejas conservan su `_ingesta_ts` original. Solo las nuevas traen uno nuevo.

In [0]:
spark.sql(f"""
SELECT DATE_TRUNC('SECOND', _ingesta_ts) AS tanda, COUNT(*) AS filas
FROM {CATALOGO}.bronze.pedidos_incremental
GROUP BY 1 ORDER BY 1
""").display()

tanda,filas
2026-09-07T01:08:08.000Z,22
2026-09-07T01:23:47.000Z,25


### Paso 4 — completar los 23 meses
La prueba incremental ya terminó: vimos una primera carga, una reejecución con **0 filas**
nuevas y la llegada de otro mes. Ahora sube a `landing/pedidos/` **todos los lotes mensuales
que todavía falten** y ejecuta la siguiente celda.

No estamos agregando pedidos posteriores a mayo de 2026. Estamos reconstruyendo, mes a mes,
la llegada histórica de las mismas **830 filas** que S01 cargó de una sola vez.

In [0]:
total_final = ingerir_pedidos()
print(f"Total después de cargar los 23 meses: {total_final:,} pedidos")
assert total_final == 830, (
    f"Se esperaban 830 pedidos y llegaron {total_final}. "
    "Revisa qué lotes mensuales faltan en landing/pedidos/."
)
print("✅ Los 23 meses quedaron cargados sin duplicados.")

Total después de cargar los 23 meses: 830 pedidos
✅ Los 23 meses quedaron cargados sin duplicados.


In [0]:
## Verificación Adicional por Clave
spark.sql(f"""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT IdPedido) AS pedidos_unicos
FROM {CATALOGO}.bronze.pedidos_incremental
""").display()

filas,pedidos_unicos
830,830


## 3 · Calidad: declarar las reglas, no confiar en ellas
Reglas de **negocio**, no de tipos. Una expectativa que falla en silencio hoy
es una alucinación de tu agente dentro de tres sesiones.

In [0]:
EXPECTATIVAS = {
    "descuento_en_rango":   "Descuento BETWEEN 0 AND 1",
    "cantidad_positiva":    "Cantidad > 0",
    "ingreso_no_negativo":  "ingreso_linea >= 0",
}

fallas = {}
for nombre, regla in EXPECTATIVAS.items():
    n = spark.sql(f"""
        SELECT COUNT(*) AS n FROM {CATALOGO}.silver.detalles_pedidos
        WHERE NOT ({regla})
    """).first()["n"]
    fallas[nombre] = n
    print(f"{'✅' if n == 0 else '❌'} {nombre:22s} violaciones: {n}")

✅ descuento_en_rango     violaciones: 0
✅ cantidad_positiva      violaciones: 0
✅ ingreso_no_negativo    violaciones: 0


### Integridad referencial
¿Hay líneas de detalle sin pedido cabecera? El generador de lotes lo garantiza —
pero **una garantía que no se verifica no es una garantía**.

In [0]:
huerfanos = spark.sql(f"""
SELECT COUNT(*) AS n
FROM {CATALOGO}.silver.detalles_pedidos d
LEFT ANTI JOIN {CATALOGO}.bronze.pedidos p USING (IdPedido)
""").first()["n"]
print(f"{'✅' if huerfanos == 0 else '❌'} líneas de detalle huérfanas: {huerfanos}")

✅ líneas de detalle huérfanas: 0


## 4 · Change Data Feed
Delta puede decirte **qué filas cambiaron entre dos versiones**, no solo el estado final.

🔗 No es plomería opcional: el **Vector Search de la sesión 4 exige CDF** para
mantener el índice sincronizado con la tabla.

In [0]:
# Habilita el registro fila por fila de inserts, updates y deletes futuros.
spark.sql(f"ALTER TABLE {CATALOGO}.silver.detalles_pedidos SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Guarda la versión actual para leer únicamente los cambios que ocurran después.
v_antes = spark.sql(f"DESCRIBE HISTORY {CATALOGO}.silver.detalles_pedidos").first()["version"]

# Provoca un UPDATE controlado para generar un evento visible en Change Data Feed.
spark.sql(f"""
UPDATE {CATALOGO}.silver.detalles_pedidos
SET Descuento = 0.25 WHERE IdPedido = (SELECT MIN(IdPedido) FROM {CATALOGO}.silver.detalles_pedidos)
""")

# table_changes devuelve las filas modificadas y la metadata del tipo de cambio.
spark.sql(f"""
SELECT IdPedido, IdProducto, Descuento, ingreso_linea, _change_type
FROM table_changes('{CATALOGO}.silver.detalles_pedidos', {v_antes + 1})
""").display()

IdPedido,IdProducto,Descuento,ingreso_linea,_change_type
10248,11,0.0,168.0,update_preimage
10248,11,0.25,168.0,update_postimage
10248,42,0.0,98.0,update_preimage
10248,42,0.25,98.0,update_postimage
10248,72,0.0,174.0,update_preimage
10248,72,0.25,174.0,update_postimage


> ⚠️ Fíjate que `ingreso_linea` **no** se recalculó: el `UPDATE` tocó `Descuento` a mano,
> por fuera del pipeline. Ésa es exactamente la razón por la que las reglas de negocio
> tienen que vivir en el pipeline y no en updates sueltos.

## 4.1 · Reconstruir Silver después de la demostración de CDF

  La demostración de Change Data Feed modificó manualmente el campo `Descuento`, pero no recalculó `ingreso_linea`. Esto dejó ambas columnas
temporalmente desalineadas. Volvemos a ejecutar la sección 1 para reconstruir `silver.detalles_pedidos` desde Bronze y aplicar nuevamente la
regla de negocio completa: el ingreso de cada línea debe calcularse usando su descuento actual.

  Después vuelve a ejecutar la celda completa de la sección 1.

## 4.2 · Confirmar que Change Data Feed continúa habilitado
 
  Al reconstruir la tabla Silver queremos asegurarnos de que Change Data Feed siga activo. Esta propiedad permitirá que procesos posteriores
identifiquen qué filas fueron insertadas, actualizadas o eliminadas sin tener que comparar la tabla completa. También será necesaria para    
mantener sincronizado el índice de Vector Search de la sesión 4.

In [0]:
spark.sql(f"""
  ALTER TABLE {CATALOGO}.silver.detalles_pedidos
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
  """)

print("✅ Change Data Feed habilitado en Silver")

✅ Change Data Feed habilitado en Silver


## 4.3 · Verificar la coherencia de la regla de negocio

  Antes de construir Gold comprobamos que `ingreso_linea` coincida con la fórmula oficial de ventas para todas las filas. Esta validación
evita publicar métricas derivadas de una tabla inconsistente. El resultado debe ser cero: cualquier valor mayor indica que el descuento y el
ingreso calculado no corresponden entre sí.

In [0]:
spark.sql(f"""
  SELECT COUNT(*) AS inconsistencias
  FROM {CATALOGO}.silver.detalles_pedidos
  WHERE ingreso_linea <> ROUND(
      PrecioUnidad * Cantidad * (1 - Descuento),
      2
  )
  """).display()

inconsistencias
0


## 5 · Gold: la capa que la IA sí puede leer
Cada tabla gold **elimina una trampa concreta** de las que vimos en el Demo 0.

### 5.1 Ventas — mata el error del descuento y la fecha equivocada
En S01 vimos que sumar `PrecioUnidad * Cantidad` infla las ventas **6,55 %** porque olvida
el descuento. También es fácil agrupar por `FechaEnvio`, aunque la venta ocurrió en
`FechaPedido`.

**Qué busca el código:** producir una fila por categoría y mes usando `ingreso_linea`, la
métrica confiable que ya calculamos en Silver, y la fecha real de la venta. El consumidor
recibe `ingreso_neto`, `unidades` y `pedidos` sin tener que reconstruir la regla.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.ventas_por_categoria_mes AS
SELECT c.NombreCategoria                        AS categoria,
       DATE_TRUNC('MONTH', p.FechaPedido)       AS mes,
       ROUND(SUM(d.ingreso_linea), 2)           AS ingreso_neto,
       SUM(d.Cantidad)                          AS unidades,
       COUNT(DISTINCT p.IdPedido)               AS pedidos
FROM {CATALOGO}.silver.detalles_pedidos d
JOIN {CATALOGO}.bronze.pedidos    p ON d.IdPedido   = p.IdPedido
JOIN {CATALOGO}.bronze.productos  pr ON d.IdProducto = pr.IdProducto
JOIN {CATALOGO}.bronze.categorias c ON pr.IdCategoria = c.IdCategoria
GROUP BY 1, 2
""")

spark.sql(f"""
COMMENT ON TABLE {CATALOGO}.gold.ventas_por_categoria_mes IS
'Ventas netas por categoría y mes. ingreso_neto YA tiene el descuento aplicado y usa FechaPedido
 (cuándo se vendió), no FechaEnvio. NO existe información de costos: esta tabla no permite
 calcular margen ni rentabilidad.'
""")
display(spark.table(f"{CATALOGO}.gold.ventas_por_categoria_mes").orderBy("mes").limit(10))

categoria,mes,ingreso_neto,unidades,pedidos
Granos y Cereales,2024-07-01T00:00:00.000Z,1256.86,83,4
Frutas y Verduras,2024-07-01T00:00:00.000Z,3868.8,156,5
Pescados y Mariscos,2024-07-01T00:00:00.000Z,2400.33,187,7
Condimentos,2024-07-01T00:00:00.000Z,1878.2,139,6
Bebidas,2024-07-01T00:00:00.000Z,3182.5,272,11
Reposteria,2024-07-01T00:00:00.000Z,5775.15,245,8
Carnes y Aves,2024-07-01T00:00:00.000Z,2661.72,76,4
Lacteos,2024-07-01T00:00:00.000Z,6838.34,304,9
Reposteria,2024-08-01T00:00:00.000Z,5006.78,170,10
Condimentos,2024-08-01T00:00:00.000Z,2296.6,154,7


> 🔑 Lee el último renglón del `COMMENT`: **declara lo que la tabla NO puede responder.**
> Eso es lo que le faltaba al Genie del Demo 0. Un modelo que lee «no existe información de
> costos» contesta *«no puedo»*. Uno que no lo lee, **improvisa un número distinto cada vez**.

### 5.2 Inventario — mata la columna olvidada
Mirar únicamente `UnidadesEnExistencia` genera alertas falsas: un producto puede tener poco
stock físico, pero ya traer mercadería en `UnidadesEnPedido`. Ignorar esa segunda columna
puede provocar una compra duplicada.

**Qué busca el código:** calcular `disponible_total = existencia + unidades en pedido`,
excluir productos suspendidos y dejar una bandera `requiere_reposicion` lista para consumir.
Al final compara cuántas alertas produciría la regla ingenua frente a la regla correcta.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.inventario_disponible AS
SELECT p.IdProducto, p.NombreProducto, c.NombreCategoria AS categoria,
       p.UnidadesEnExistencia, p.UnidadesEnPedido, p.NivelNuevoPedido,
       p.UnidadesEnExistencia + p.UnidadesEnPedido AS disponible_total,
       (p.UnidadesEnExistencia + p.UnidadesEnPedido) < p.NivelNuevoPedido AS requiere_reposicion
FROM {CATALOGO}.bronze.productos p
JOIN {CATALOGO}.bronze.categorias c ON p.IdCategoria = c.IdCategoria
WHERE p.Suspendido = 0
""")

spark.sql(f"""
COMMENT ON TABLE {CATALOGO}.gold.inventario_disponible IS
'Disponibilidad real de producto. disponible_total suma existencias MÁS UnidadesEnPedido (mercadería
 ya comprada al proveedor). Un producto sin stock físico pero con unidades en pedido NO requiere
 reposición. Excluye productos suspendidos.'
""")

spark.sql(f"""
SELECT COUNT(*) FILTER (WHERE UnidadesEnExistencia < NivelNuevoPedido) AS alerta_ingenua,
       COUNT(*) FILTER (WHERE requiere_reposicion)                     AS alerta_correcta
FROM {CATALOGO}.gold.inventario_disponible
""").display()

alerta_ingenua,alerta_correcta
17,2


> Los dos números no coinciden. **La diferencia son órdenes de compra que no hacían falta.**

### 5.3 Meses completos — mata la comparación injusta
El dataset termina el 6 de mayo de 2026: mayo tiene solo **14 pedidos**, frente a **74** en
abril. Compararlos directamente haría parecer que el negocio se desplomó, cuando en realidad
estamos comparando un mes parcial contra uno completo.

**Qué busca el código:** agrupar los pedidos por mes, encontrar el último mes disponible y
marcarlo como `mes_completo = false`. Los análisis temporales pueden filtrar
`mes_completo = true` y evitar conclusiones falsas por períodos inconclusos.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.ventas_mes_completo AS
WITH por_mes AS (
  SELECT DATE_TRUNC('MONTH', FechaPedido) AS mes,
         MAX(FechaPedido)                 AS ultimo_pedido,
         COUNT(*)                         AS pedidos
  FROM {CATALOGO}.bronze.pedidos GROUP BY 1
), corte AS (SELECT MAX(mes) AS mes_final FROM por_mes)
SELECT m.mes, m.pedidos, m.ultimo_pedido,
       m.mes < c.mes_final AS mes_completo
FROM por_mes m CROSS JOIN corte c
""")

spark.sql(f"""
COMMENT ON TABLE {CATALOGO}.gold.ventas_mes_completo IS
'Pedidos por mes con la marca mes_completo. El último mes del dataset está PARCIAL: compararlo
 contra un mes completo produce una caída falsa. Toda comparación temporal debe filtrar
 mes_completo = true.'
""")
display(spark.table(f"{CATALOGO}.gold.ventas_mes_completo").orderBy(F.col("mes").desc()).limit(4))

mes,pedidos,ultimo_pedido,mes_completo
2026-05-01T00:00:00.000Z,14,2026-05-06,false
2026-04-01T00:00:00.000Z,74,2026-04-30,true
2026-03-01T00:00:00.000Z,73,2026-03-31,true
2026-02-01T00:00:00.000Z,54,2026-02-27,true


> 🎯 Mira el último mes contra el anterior. Esa caída **no es del negocio: es un mes que no
> terminó.** Gold lo declara con una columna en vez de esperar que el analista se dé cuenta.

## 6 · Tu entregable
1. Pipeline incremental corriendo con **los 23 meses** cargados en tandas
2. **Una regla de calidad propia**, validada con SQL como las reglas de la sección 3
3. **Una tabla gold más**, con su regla de negocio en el `COMMENT`

> `CONSTRAINT ... EXPECT`, Lakeflow Pipelines y Jobs quedan como ampliación opcional en
> `notebook-append.py`; no son requisito para continuar con la sesión 3.

---
**La semana que viene:** ya tenemos datos confiables. Entra el primer LLM — y le volvemos
a hacer al Genie la pregunta del margen, pero apuntando a **nuestro gold**.

**Apaga el compute.**

### Entregable 1: confirmar el pipeline incremental

In [0]:
resultado_pipeline = spark.sql(f"""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT IdPedido) AS pedidos_unicos,
       COUNT(DISTINCT DATE_TRUNC('MONTH', TO_DATE(FechaPedido))) AS meses
FROM {CATALOGO}.bronze.pedidos_incremental
""").first()

assert resultado_pipeline["filas"] == 830
assert resultado_pipeline["pedidos_unicos"] == 830
assert resultado_pipeline["meses"] == 23
print("✅ 830 pedidos únicos distribuidos en 23 meses")


✅ 830 pedidos únicos distribuidos en 23 meses


### Entregable 2: regla de calidad propia

In [0]:
regla_propia = """
FechaEnvio IS NOT NULL
AND TO_DATE(FechaEnvio) < TO_DATE(FechaPedido)
"""

violaciones_regla_propia = spark.sql(f"""
SELECT COUNT(*) AS n
FROM {CATALOGO}.bronze.pedidos_incremental
WHERE {regla_propia}
""").first()["n"]

print(
    f"{'✅' if violaciones_regla_propia == 0 else '❌'} "
    f"envío anterior al pedido: {violaciones_regla_propia} violaciones"
)
assert violaciones_regla_propia == 0, "Hay pedidos enviados antes de ser creados"

✅ envío anterior al pedido: 0 violaciones


### Entregable 3: tabla Gold propia

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOGO}.gold.desempeno_despacho_transportistas AS
SELECT t.NombreCompania AS transportista,
       COUNT(*) AS pedidos_enviados,
       ROUND(AVG(DATEDIFF(TO_DATE(p.FechaEnvio), TO_DATE(p.FechaPedido))), 2)
           AS dias_promedio_preparacion,
       MAX(DATEDIFF(TO_DATE(p.FechaEnvio), TO_DATE(p.FechaPedido)))
           AS peor_caso_dias,
       ROUND(SUM(CAST(p.Cargo AS DECIMAL(12,2))), 2) AS cargo_total
FROM {CATALOGO}.bronze.pedidos_incremental p
JOIN {CATALOGO}.bronze.transportistas t
  ON p.IdTransportista = t.IdTransportista
WHERE p.FechaEnvio IS NOT NULL
GROUP BY t.NombreCompania
""")

spark.sql(f"""
COMMENT ON TABLE {CATALOGO}.gold.desempeno_despacho_transportistas IS
'Una fila por transportista con pedidos ya enviados. dias_promedio_preparacion mide días entre
 FechaPedido y FechaEnvio; no mide tránsito ni entrega real. cargo_total suma el campo Cargo y
 no representa margen, rentabilidad ni costo operativo del transportista.'
""")

display(
    spark.table(f"{CATALOGO}.gold.desempeno_despacho_transportistas")
    .orderBy("dias_promedio_preparacion")
)

transportista,pedidos_enviados,dias_promedio_preparacion,peor_caso_dias,cargo_total
Envios Federales,249,7.47,35,20363.10
Expreso Veloz,245,8.57,37,16035.16
Paquetes Unidos,315,9.23,37,27556.76


#### Verifica el resultado y el comentario

In [0]:
spark.sql(f"""
SELECT COUNT(*) AS transportistas,
       SUM(pedidos_enviados) AS pedidos_enviados
FROM {CATALOGO}.gold.desempeno_despacho_transportistas
""").display()

spark.sql(f"""
DESCRIBE TABLE EXTENDED {CATALOGO}.gold.desempeno_despacho_transportistas
""").display()

transportistas,pedidos_enviados
3,809


col_name,data_type,comment
transportista,string,null
pedidos_enviados,bigint,null
dias_promedio_preparacion,double,null
peor_caso_dias,int,null
cargo_total,"decimal(23,2)",null
,,
# Delta Statistics Columns,,
Column Names,"peor_caso_dias, dias_promedio_preparacion, transportista, pedidos_enviados, cargo_total",
Column Selection Method,first-32,
,,


### Verificación final

In [0]:
tablas_esperadas = [
    f"{CATALOGO}.silver.detalles_pedidos",
    f"{CATALOGO}.bronze.pedidos_incremental",
    f"{CATALOGO}.gold.ventas_por_categoria_mes",
    f"{CATALOGO}.gold.inventario_disponible",
    f"{CATALOGO}.gold.ventas_mes_completo",
    f"{CATALOGO}.gold.desempeno_despacho_transportistas",
]

for tabla in tablas_esperadas:
    assert spark.catalog.tableExists(tabla), f"Falta {tabla}"
    print(f"✅ {tabla}")

filas = spark.table(f"{CATALOGO}.bronze.pedidos_incremental").count()
unicos = spark.sql(f"""
SELECT COUNT(DISTINCT IdPedido) AS n
FROM {CATALOGO}.bronze.pedidos_incremental
""").first()["n"]
inconsistencias = spark.sql(f"""
SELECT COUNT(*) AS n
FROM {CATALOGO}.silver.detalles_pedidos
WHERE ingreso_linea <> ROUND(PrecioUnidad * Cantidad * (1 - Descuento), 2)
""").first()["n"]
cdf = spark.sql(f"""
SHOW TBLPROPERTIES {CATALOGO}.silver.detalles_pedidos ('delta.enableChangeDataFeed')
""").first()["value"]

assert filas == 830 and unicos == 830
assert inconsistencias == 0
assert cdf.lower() == "true"
print("🎉 Sesión 2 completa: incrementalidad, calidad, Gold y CDF verificados")

✅ neptuno_emerson_suarez.silver.detalles_pedidos
✅ neptuno_emerson_suarez.bronze.pedidos_incremental
✅ neptuno_emerson_suarez.gold.ventas_por_categoria_mes
✅ neptuno_emerson_suarez.gold.inventario_disponible
✅ neptuno_emerson_suarez.gold.ventas_mes_completo
✅ neptuno_emerson_suarez.gold.desempeno_despacho_transportistas
🎉 Sesión 2 completa: incrementalidad, calidad, Gold y CDF verificados
